# 16. Advanced Time-Series & Intervals: Beginner Guide

### 📌 Overview
Master **16. Advanced Time-Series & Intervals: Beginner Guide** with concise, zero-fluff bullet points and executable code on real Fintech records ([raw_transactions.csv](file:///data/raw_transactions.csv)).

### 📚 Key Concepts Covered in this Notebook:
- **Range Generation**: Covers `pd.period_range()`, `pd.date_range()`, and `pd.interval_range()`.
- **Business Day Logic**: Covers business day offsets (`'B'`, `'BM'`) and `pd.bdate_range()`.
- **Continuous Binning**: Covers `pd.cut()` and `pd.qcut()`.


In [1]:
# Setup imports & dataset loading
import pandas as pd
import numpy as np
import sys
import time
import os
import matplotlib.pyplot as plt

# Load raw transactions dataset
csv_path = 'data/raw_transactions.csv' if os.path.exists('data/raw_transactions.csv') else '../data/raw_transactions.csv'
df = pd.read_csv(csv_path)
print(f"Pandas Version: {pd.__version__}")
print(f"Loaded raw_transactions.csv: {df.shape[0]} rows, {df.shape[1]} columns")

Pandas Version: 2.2.2
Loaded raw_transactions.csv: 15000 rows, 11 columns


### 🔹 Datetime Ranges with `pd.date_range()`
- **What it does:** Generates daily settlement audit timestamp sequences.
- **Syntax:** `pd.date_range(start='2025-01-01', periods=10, freq='D')`
- **Operation:** `settlement_dates = pd.date_range(start='2025-01-01', periods=7, freq='D')`
- **Key Note:** Ensure your column is converted to datetime with `pd.to_datetime()` before calling `.dt` properties like `.dt.year` or `.dt.day_name()`.

In [2]:
settlement_dates = pd.date_range(start='2025-01-01', periods=7, freq='D')
print('Generated Settlement Dates:\n', settlement_dates)

Generated Settlement Dates:
 DatetimeIndex(['2025-01-01', '2025-01-02', '2025-01-03', '2025-01-04',
               '2025-01-05', '2025-01-06', '2025-01-07'],
              dtype='datetime64[ns]', freq='D')


### 🔹 Accounting Periods with `pd.period_range()`
- **What it does:** Generates monthly fiscal reporting periods.
- **Syntax:** `pd.period_range(start='2025-01', periods=6, freq='M')`
- **Operation:** `fiscal_periods = pd.period_range(start='2025-01', periods=6, freq='M')`
- **Key Note:** Inspect your data types early with `df.dtypes` to ensure numeric values weren't accidentally parsed as text.

In [3]:
fiscal_periods = pd.period_range(start='2025-01', periods=6, freq='M')
print('Fiscal Period Spans:\n', fiscal_periods)

Fiscal Period Spans:
 PeriodIndex(['2025-01', '2025-02', '2025-03', '2025-04', '2025-05', '2025-06'], dtype='period[M]')


### 🔹 Mathematical Intervals with `pd.interval_range()`
- **What it does:** Constructs regular spending tier intervals.
- **Syntax:** `pd.interval_range(start=0, end=1500, periods=5, closed='left')`
- **Key Note:** Inspect your data types early with `df.dtypes` to ensure numeric values weren't accidentally parsed as text.

In [4]:
spend_intervals = pd.interval_range(start=0, end=1500, periods=5, closed='left')
print('Spend Intervals:\n', spend_intervals)

Spend Intervals:
 IntervalIndex([[0, 300), [300, 600), [600, 900), [900, 1200), [1200, 1500)], dtype='interval[int64, left]')


### 🔹 Business Day Calendars with `pd.bdate_range()`
- **What it does:** Generates banking settlement dates automatically skipping weekends.
- **Syntax:** `pd.bdate_range(start='2025-01-01', periods=5, freq='B')`
- **Operation:** `banking_days = pd.bdate_range(start='2025-01-03', periods=5, freq='B')`
- **Key Note:** Ensure your column is converted to datetime with `pd.to_datetime()` before calling `.dt` properties like `.dt.year` or `.dt.day_name()`.

In [5]:
banking_days = pd.bdate_range(start='2025-01-03', periods=5, freq='B')
print('Banking Business Days (skips weekends):\n', banking_days)

Banking Business Days (skips weekends):
 DatetimeIndex(['2025-01-03', '2025-01-06', '2025-01-07', '2025-01-08',
               '2025-01-09'],
              dtype='datetime64[ns]', freq='B')


### 🔹 Equal-Width Discretization with `pd.cut()`
- **What it does:** Discretizes transaction amounts into fixed spending brackets (Micro, Small, Medium, Large).
- **Syntax:** `pd.cut(df['transaction_amount'], bins=[0, 50, 200, 800, 2000], labels=['Micro', 'Small', 'Medium', 'Large'])`
- **Operation:** `df_valid = df.dropna(subset=['transaction_amount']).copy()`
- **Key Note:** Inspect your data types early with `df.dtypes` to ensure numeric values weren't accidentally parsed as text.

In [6]:
df_valid = df.dropna(subset=['transaction_amount']).copy()
df_valid['spend_tier'] = pd.cut(df_valid['transaction_amount'], bins=[0, 50, 200, 800, 5000], labels=['Micro', 'Small', 'Medium', 'Large'])
print('Discretized Spend Tier Breakdown:\n', df_valid['spend_tier'].value_counts())

Discretized Spend Tier Breakdown:
 spend_tier
Large     8551
Medium    4316
Small     1073
Micro      311
Name: count, dtype: int64


### 🔹 Equal-Frequency Quantile Binning with `pd.qcut()`
- **What it does:** Bins transactions into 4 balanced revenue quartiles (`Q1`, `Q2`, `Q3`, `Q4`).
- **Syntax:** `pd.qcut(df['transaction_amount'], q=4, labels=['Q1', 'Q2', 'Q3', 'Q4'])`
- **Key Note:** Inspect your data types early with `df.dtypes` to ensure numeric values weren't accidentally parsed as text.

In [7]:
df_valid['spend_quartile'] = pd.qcut(df_valid['transaction_amount'], q=4, labels=['Q1', 'Q2', 'Q3', 'Q4'])
print('Balanced Quartile Counts:\n', df_valid['spend_quartile'].value_counts())

Balanced Quartile Counts:
 spend_quartile
Q1    3563
Q2    3563
Q4    3563
Q3    3562
Name: count, dtype: int64


## 💡 Real-World Practice & Scenarios
Practical scenarios and common data engineering questions explained with real examples.


### 🔍 Scenario: Q1: Fraud Rate Distribution Across Spend Quartiles
- **Objective:** Q1: Fraud Rate Distribution Across Spend Quartiles
- **Approach:** Analyze whether the fraud rate increases exponentially in the highest spend quartile (Q4).
- **Syntax:** `df.groupby('spend_quartile')['is_fraud'].agg(['count', 'mean'])`

In [8]:
quartile_fraud = df_valid.groupby('spend_quartile', observed=False)['is_fraud'].agg(['count', 'mean'])
print('Fraud Rate Breakdown Across Spend Quartiles:\n', quartile_fraud)

Fraud Rate Breakdown Across Spend Quartiles:
                 count      mean
spend_quartile                 
Q1               3563  0.011507
Q2               3563  0.007578
Q3               3562  0.009264
Q4               3563  0.423800
